In [ ]:
import os
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)


@dataclass(frozen=True)
class Provider:
    name: str
    env_var: str
    base_url: str | None
    model: str


PROVIDERS = [
    Provider(
        name="Groq",
        env_var="GROQ_API_KEY",
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-120b",
    ),
    Provider(
        name="OpenRouter",
        env_var="OPENROUTER_API_KEY",
        base_url="https://openrouter.ai/api/v1",
        model="openai/gpt-4o-mini",
    ),
    Provider(
        name="OpenAI",
        env_var="OPENAI_API_KEY",
        base_url=None,
        model="gpt-4o-mini",
    ),
]


def select_provider() -> Provider:
    selected = os.getenv("LLM_PROVIDER", "").strip()

    if selected:
        for provider in PROVIDERS:
            if provider.name.lower() == selected.lower():
                if not os.getenv(provider.env_var):
                    raise RuntimeError(
                        f"Missing {provider.env_var} in your .env file."
                    )
                return provider

        available = ", ".join(p.name for p in PROVIDERS)
        raise RuntimeError(
            f"Unknown provider '{selected}'. Choose from: {available}"
        )

    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    raise RuntimeError(
        "No API key found. Configure GROQ_API_KEY, "
        "OPENROUTER_API_KEY, or OPENAI_API_KEY in .env."
    )


def build_client(provider: Provider) -> OpenAI:
    return OpenAI(
        api_key=os.getenv(provider.env_var),
        **({"base_url": provider.base_url} if provider.base_url else {}),
    )


provider = select_provider()

print("Provider:", provider.name)
print("Model:", provider.model)
print("API key found:", bool(os.getenv(provider.env_var)))

Provider: OpenRouter
Model: openai/gpt-4o-mini
API key found: True


In [ ]:
def chat_reply(messages: list[dict]) -> str:
    provider = select_provider()
    client = build_client(provider)

    result = client.chat.completions.create(
        model=provider.model,
        messages=messages,
        max_tokens=1000,
    )

    reply = result.choices[0].message.content

    if not reply:
        return "The model returned an empty response."

    return reply

In [ ]:
# Store the conversation history
conversation = []


def send_message(user_input: str) -> str:
    """Send a message and retain the conversation history."""

    if not user_input.strip():
        return "Please enter a message."

    # 1. Save the user's message
    conversation.append({
        "role": "user",
        "content": user_input,
    })

    try:
        # 2. Send the entire conversation to the LLM
        reply = chat_reply(conversation)

        # 3. Save the assistant's reply
        conversation.append({
            "role": "assistant",
            "content": reply,
        })

        return reply

    except Exception:
        # Remove the unanswered user message if the request fails
        conversation.pop()
        raise


print("Stateful chatbot is ready!")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break

    try:
        response = send_message(user_input)
        print(f"Assistant: {response}\n")

    except Exception as error:
        print(f"Error: {error}\n")

Stateful chatbot is ready!
Type 'exit' to stop.

Error: Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}

Error: Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}

Error: Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}

Chat ended.


In [ ]:
print("Conversation:", conversation)
print("Number of messages:", len(conversation))

Conversation: []
Number of messages: 0
